In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)

Beschikbare datasets:
['notebooks', 'datasets', 'competitions']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/gith-l/src/hms_sklearn_baseline.py


In [2]:
from pathlib import Path

for folder in Path("/kaggle/input").iterdir():
    print(f"\nDataset: {folder.name}")

    for file in folder.rglob("*"):
        if file.is_file() and (
            "baseline" in file.name.lower()
            or "optimal" in file.name.lower()
            or file.suffix.lower() in [".ipynb", ".py"]
        ):
            print("  ", file)


Dataset: notebooks
   /kaggle/input/notebooks/hannaleysen/basline-optimalisatie/__notebook__.ipynb

Dataset: datasets
   /kaggle/input/datasets/hannaleysen/gith-l/Kaggle_notebook.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/Test_push_pandas.py
   /kaggle/input/datasets/hannaleysen/gith-l/template.py
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-02-KNN.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-03-KNN+PCA.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-04-LDA.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-05-linear-regression-newton.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-06-MLP.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-07-MLP+SDG.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/hms-experiment-08-XGBoost-full-mode.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/stap-1-baseli

In [3]:
# geoptimaliseerde baseline inlezen 


import nbformat
import ast
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/hannaleysen/basline-optimalisatie/__notebook__.ipynb")
nb = nbformat.read(NOTEBOOK_PATH, as_version=4)

# Haal ALLEEN de functie-definities op uit cel 5 (zodat er geen 'df' wordt verwacht)
tree = ast.parse(nb.cells[5].source)
selected_nodes = [node for node in tree.body if isinstance(node, ast.FunctionDef)]

clean_tree = ast.Module(body=selected_nodes, type_ignores=[])
ast.fix_missing_locations(clean_tree)
exec(compile(clean_tree, "<optimized_features>", "exec"), globals())

print("Geoptimaliseerde featurefuncties veilig geladen!")

print("build_features:", build_features)
print("window_features:", window_features)
print("spectrogram_features:", spectrogram_features)
print("band_columns:", band_columns)

Geoptimaliseerde featurefuncties veilig geladen!
build_features: <function build_features at 0x7bdf76b8b920>
window_features: <function window_features at 0x7bdf76b8b7e0>
spectrogram_features: <function spectrogram_features at 0x7bdf76b8b560>
band_columns: <function band_columns at 0x7bdf76b8b4c0>


In [4]:
import inspect

print(inspect.signature(build_features))
print(inspect.signature(window_features))
print(inspect.signature(spectrogram_features))

(meta, spec_dir, is_test=False)
(spec, cols)
(path, windows, is_test)


In [5]:
from pathlib import Path
import importlib.util

scripts = list(Path("/kaggle/input").rglob("hms_sklearn_baseline.py"))

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht 1 baseline-bestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]

spec = importlib.util.spec_from_file_location("hms_baseline", SCRIPT_PATH)
baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline geladen:", SCRIPT_PATH)

Baseline geladen: /kaggle/input/datasets/hannaleysen/gith-l/src/hms_sklearn_baseline.py


In [6]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [7]:
import os
import time
import warnings
import ast
import nbformat
import numpy as np
import pandas as pd
import inspect

from pathlib import Path
from joblib import Parallel, delayed

In [8]:
# DATA INLADEN EN GEOPTIMALISEERDE FEATURES
N_JOBS = 2
REGIONS = ["LL", "RL", "LP", "RP"]
BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 20)}


# Pad naar de HMS-competitiedataset
DATA_DIR = Path(
    "/kaggle/input/competitions/hms-harmful-brain-activity-classification")

assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

# 1. Labels inladen (oude baseline blijft hiervoor gebruikt)
df = baseline.load_labels(DATA_DIR, max_specs=None, seed=42)
print(df.shape, "EEGs")
print(df.patient_id.nunique(), "verschillende patiënten")
print(df.spectrogram_id.nunique(), "spectrogrammen")

# 2. NIEUW: geoptimaliseerde feature-extractie
X_df = build_features(df, DATA_DIR / "train_spectrograms")

# 3. Controle van het aantal features
feats = [c for c in X_df.columns if c != "eeg_id"]
print("\nAantal geoptimaliseerde features:", len(feats))
assert len(feats) == 192, (f"Verwacht 192 features, maar gevonden: {len(feats)}")

# 4. Labels en features samenvoegen
df = df.merge(X_df, on="eeg_id", how="inner")

# 5. Matrix voor XGBoost
X = baseline.to_matrix(df, feats)

# 6. Doelvariabelen en groepen
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)
groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()

print("\nData klaar voor XGBoost!")
print("Aantal EEG-records:", len(X))
print("Aantal features:", X.shape[1])
print("Aantal klassen:", Y.shape[1])
print("Vorm van X:", X.shape)
print("Vorm van Y:", Y.shape)

print("\nEerste 20 features:")
for i, feature in enumerate(feats[:20]):
    print(f"{i+1}. {feature}")


(17089, 12) EEGs
1950 verschillende patiënten
11138 spectrogrammen
11138 spectrograms -> 17089 feature rows in 428s

Aantal geoptimaliseerde features: 192

Data klaar voor XGBoost!
Aantal EEG-records: 17089
Aantal features: 192
Aantal klassen: 6
Vorm van X: (17089, 192)
Vorm van Y: (17089, 6)

Eerste 20 features:
1. LL_delta_mean
2. LL_delta_std
3. LL_delta_min
4. LL_delta_max
5. LL_delta_median
6. LL_delta_q25
7. LL_delta_q75
8. LL_delta_iqr
9. LL_delta_trend
10. LL_theta_mean
11. LL_theta_std
12. LL_theta_min
13. LL_theta_max
14. LL_theta_median
15. LL_theta_q25
16. LL_theta_q75
17. LL_theta_iqr
18. LL_theta_trend
19. LL_alpha_mean
20. LL_alpha_std


In [9]:
# nog extra afgeleide features toevoegen voor een betere score

# LINKS RECHTS ASYMMETRIE + BANDCONTRASTEN
def add_derived_features(X_df):
    regions = ["LL", "RL", "LP", "RP"]
    bands = ["delta", "theta", "alpha", "beta"]
    stats = ["mean", "std", "max", "median"]
    new = {}
    for b in bands:
        for s in stats:
            cols = [f"{r}_{b}_{s}" for r in regions]
            if not all(c in X_df.columns for c in cols):
                continue
            V = X_df[cols].to_numpy(dtype=np.float64)
            LL, RL, LP, RP = V.T
            new[f"x_hemi_lat_{b}_{s}"] = LL - RL            # links vs rechts (lateraal)
            new[f"x_hemi_par_{b}_{s}"] = LP - RP            # links vs rechts (parasagittaal)
            new[f"x_abs_hemi_{b}_{s}"] = np.abs(LL - RL) + np.abs(LP - RP)
            new[f"x_left_ll-lp_{b}_{s}"] = LL - LP
            new[f"x_right_rl-rp_{b}_{s}"] = RL - RP
            new[f"x_reg_mean_{b}_{s}"] = V.mean(axis=1)     # globaal niveau
            new[f"x_reg_std_{b}_{s}"] = V.std(axis=1)       # spreiding tussen regio's
            new[f"x_reg_max_{b}_{s}"] = V.max(axis=1)
            new[f"x_reg_min_{b}_{s}"] = V.min(axis=1)
    # contrasten tussen frequentiebanden binnen een regio
    for r in regions:
        for s in ["mean", "max", "median"]:
            c = {b: f"{r}_{b}_{s}" for b in bands}
            if not all(v in X_df.columns for v in c.values()):
                continue
            new[f"x_{r}_theta-delta_{s}"] = X_df[c["theta"]] - X_df[c["delta"]]
            new[f"x_{r}_alpha-theta_{s}"] = X_df[c["alpha"]] - X_df[c["theta"]]
            new[f"x_{r}_beta-alpha_{s}"] = X_df[c["beta"]] - X_df[c["alpha"]]
            new[f"x_{r}_fast-slow_{s}"] = (X_df[c["alpha"]] + X_df[c["beta"]]) / 2 \
                                        - (X_df[c["delta"]] + X_df[c["theta"]]) / 2
    return pd.concat([X_df, pd.DataFrame(new, index=X_df.index)], axis=1)


X_df = add_derived_features(X_df)
feats = [c for c in X_df.columns if c != "eeg_id"]
new_cols = [c for c in feats if c not in df.columns]
df = df.merge(X_df[["eeg_id"] + new_cols], on="eeg_id", how="inner")

X = baseline.to_matrix(df, feats)
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)
groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()
print("Nieuw aantal features:", X.shape[1])   # ~384

Nieuw aantal features: 384


In [10]:
# Bibliotheken importeren 
import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, ParameterGrid


In [11]:
# label-ruis door onbetrouwbare rijen weg 
def expand_soft_labels(X, Y, row_w=None):
    n, k = Y.shape
    X_repeated = np.repeat(X, k, axis=0)
    y_repeated = np.tile(np.arange(k), n)
    weights = Y.reshape(-1).astype(np.float64)
    if row_w is not None:                      # NIEUW: betrouwbaarheid van de rij
        weights = weights * np.repeat(row_w, k)
    keep = weights > 0
    return X_repeated[keep], y_repeated[keep], weights[keep]


def vote_weights(num_votes, power=0.5, cap=20):
    """Rijgewicht uit aantal stemmen; power=0 -> geen gewichten, 1 -> lineair."""
    w = np.minimum(num_votes, cap) ** power
    return w / w.mean()


XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 6,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "max_bin": 128,
    "max_depth": 4,
    "eta": 0.04,
    "subsample": 0.7,
    "colsample_bytree": 0.5,
    "min_child_weight": 5,
    "lambda": 10.0,
    "alpha": 0.5,
    "seed": 42,
    "nthread": 2,
}


def train_xgboost(X_train, Y_train, X_valid, Y_valid,
                  w_train=None, w_valid=None, params=None, seed=42):
    params = dict(params or XGB_PARAMS)
    params["seed"] = seed
    Xt, yt, wt = expand_soft_labels(X_train, Y_train, w_train)
    Xv, yv, wv = expand_soft_labels(X_valid, Y_valid, w_valid)
    dtrain = xgb.DMatrix(Xt, label=yt, weight=wt)
    dvalid = xgb.DMatrix(Xv, label=yv, weight=wv)
    return xgb.train(params, dtrain, num_boost_round=3000,
                     evals=[(dvalid, "valid")],
                     early_stopping_rounds=100, verbose_eval=False)


def predict_xgboost(model, X):
    d = xgb.DMatrix(X)
    try:
        preds = model.predict(d, iteration_range=(0, model.best_iteration + 1))
    except AttributeError:          # model zonder early stopping
        preds = model.predict(d)
    preds = preds.reshape(-1, 6)
    preds = np.clip(preds, 1e-15, 1.0)
    return preds / preds.sum(axis=1, keepdims=True)


def run_cv(X, Y, groups, row_w, params, n_folds=5, seed=42, verbose=True):
    oof = np.zeros_like(Y, dtype=np.float64)
    models, iters = [], []
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=n_folds).split(X, Y, groups), 1):
        m = train_xgboost(X[tr], Y[tr], X[va], Y[va], row_w[tr], row_w[va], params, seed)
        oof[va] = predict_xgboost(m, X[va])
        models.append(m)
        iters.append(m.best_iteration + 1)
        if verbose:
            print(f"  seed {seed} fold {fold}/{n_folds}: KL {kl_score(Y[va], oof[va]):.4f} "
                  f"(iter {m.best_iteration + 1})")
    return oof, models, iters

In [12]:
# functie om KL-score te berekenen 
def kl_score(y_true, y_pred, eps=1e-15):
    """
    Bereken de gemiddelde KL-divergentie.
    Een lagere score is beter.
    """

    p = np.clip(y_pred, eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)

    t = np.asarray(y_true)

    terms = np.zeros_like(t, dtype=np.float64)
    mask = t > 0

    terms[mask] = (
        t[mask] * np.log(t[mask] / p[mask])
    )

    return np.mean(np.sum(terms, axis=1))

In [13]:
# Cross-validatie met early stopping

N_FOLDS = 3
xgb_oof = np.zeros_like(Y, dtype=np.float64)
fold_scores = []
best_iterations = []

cv = GroupKFold(n_splits=N_FOLDS)

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, Y, groups), start=1):
    print(f"Start XGBoost fold {fold}/{N_FOLDS}")

    # Train- en validatiedata apart houden
    X_train = X[train_idx]
    Y_train = Y[train_idx]
    X_valid = X[valid_idx]
    Y_valid = Y[valid_idx]

    # Train met early stopping
    model = train_xgboost(X_train, Y_train, X_valid,Y_valid)

    # Beste aantal iteraties opslaan
    best_iterations.append(model.best_iteration + 1)

    # Voorspellingen maken met de beste iteratie
    predictions = predict_xgboost(model, X_valid)
    xgb_oof[valid_idx] = predictions
    score = kl_score(Y_valid, predictions)
    fold_scores.append(score)

    print(f"Fold {fold} KL: {score:.6f}")
    print(f"Beste iteratie: {model.best_iteration + 1}")

print("Gemiddelde fold KL:", np.mean(fold_scores))
print("Totale OOF KL:", kl_score(Y, xgb_oof))
print("Gemiddelde beste iteratie:", np.mean(best_iterations))


# --- Evaluatie ---
hi = num_votes >= 10
print("\nKL per rij, alle rijen      :", kl_score(Y, xgb_oof))
print("KL per rij, >=10 stemmen    :", kl_score(Y[hi], xgb_oof[hi]))


Start XGBoost fold 1/3
Fold 1 KL: 0.911575
Beste iteratie: 508
Start XGBoost fold 2/3
Fold 2 KL: 0.867255
Beste iteratie: 443
Start XGBoost fold 3/3
Fold 3 KL: 0.848724
Beste iteratie: 547
Gemiddelde fold KL: 0.8758515415865648
Totale OOF KL: 0.8758536396554375
Gemiddelde beste iteratie: 499.3333333333333

KL per rij, alle rijen      : 0.8758536396554375
KL per rij, >=10 stemmen    : 0.841973853984259


In [14]:
# ===== FINAL XGBoost: fit op ALLE trainingsdata, voorspel test, schrijf submission.csv =====

# 1. Model trainen op alle trainingsdata (X en Y zijn al de volledige set)
final_rounds = int(np.median(best_iterations))
X_exp, y_exp, weights = expand_soft_labels(X, Y)
dtrain = xgb.DMatrix(X_exp, label=y_exp, weight=weights)
final_model = xgb.train(params=XGB_PARAMS, dtrain=dtrain, num_boost_round=final_rounds, verbose_eval=False)

# 2. Testfeatures (zelfde stappen als in de baseline)
test = pd.read_csv(DATA_DIR / "test.csv")
T_df = build_features(test, DATA_DIR / "test_spectrograms", is_test=True)
T_df = add_derived_features(T_df)                     
T_df = test[["eeg_id"]].merge(T_df, on="eeg_id", how="left")
T = baseline.to_matrix(T_df, feats)
assert T.shape[1] == len(feats)


# 3. Voorspellen en submission schrijven
probs = predict_xgboost(final_model, T)

sub = pd.DataFrame({"eeg_id": T_df["eeg_id"]})
sub[baseline.CLASSES] = probs

assert len(sub) == len(test)
assert not sub.isna().any().any()
assert np.allclose(sub[baseline.CLASSES].sum(axis=1), 1.0)

sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv", sub.shape)
sub.head()

1 spectrograms -> 1 feature rows in 0s
Saved submission.csv (1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.081727,0.036998,0.002844,0.195826,0.083155,0.59945
